# Tool calling

A *tool* is just a normal Python function we let the model use.
The model doesn't run it — it tells us *which* function and *what arguments*, and we run it.

## Setup
Run these two cells first: install the libraries, then create the `llm`.

In [ ]:
# Run once. Installs the workshop libraries.
%pip install -q langchain langchain-core langchain-openai langgraph pydantic

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

# Paste your OpenRouter key when prompted (get one at https://openrouter.ai/keys).
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

# OpenRouter uses the OpenAI API, so we use ChatOpenAI and just change the URL.
# Pass the key explicitly: ChatOpenAI does NOT read OPENROUTER_API_KEY on its own.
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",                       # any model from openrouter.ai/models
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)
print("LLM ready:", llm.model_name)

## Define a tool

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

llm_with_tools = llm.bind_tools([add])

## The model asks us to run the tool

In [ ]:
answer = llm_with_tools.invoke("What is 124 + 388?")
answer.tool_calls

## So we run the function ourselves

In [ ]:
args = answer.tool_calls[0]["args"]
add.invoke(args)